In [1]:
import csv
import json
import logging

logging.basicConfig(level=logging.INFO)

def make_report(csv_path: str, json_path: str) -> int:
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            students = list(reader)
    except FileNotFoundError:
        logging.warning(f"CSV 파일이 없습니다: {csv_path}")
        return 0
    except UnicodeDecodeError as e:
        logging.error(f"CSV 인코딩 오류: {e}")
        return 0

    results = []
    for row in students:
        name     = row["이름"]
        sid      = row["학번"]
        mid_raw  = row["중간"].strip()
        fin_raw  = row["기말"].strip()
        asgn_raw = row["과제"].strip()

        if mid_raw == "" or fin_raw == "" or asgn_raw == "":
            avg   = None
            grade = None
            logging.info(f"{name}: 평균 None, 등급 None (결측값 있음)")
        else:
            mid, fin, asgn = float(mid_raw), float(fin_raw), float(asgn_raw)
            avg = mid * 0.3 + fin * 0.5 + asgn * 0.2
            if avg >= 90:
                grade = "A"
            elif avg >= 80:
                grade = "B"
            elif avg >= 70:
                grade = "C"
            else:
                grade = "F"
            logging.info(f"{name}: 평균 {avg:.1f}, 등급 {grade}")

        results.append({
            "이름": name,
            "학번": sid,
            "점수": {
                "중간":  None if mid_raw  == "" else float(mid_raw),
                "기말":  None if fin_raw  == "" else float(fin_raw),
                "과제":  None if asgn_raw == "" else float(asgn_raw),
            },
            "평균": round(avg, 1) if avg is not None else None,
            "등급": grade,
        })

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    return len(results)

make_report("scores.csv", "report.json")

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None (결측값 있음)


4

결측값은 strip() 후 빈 문자열 여부로 판단하며, 하나라도 비어 있으면 해당 학생의 평균과 등급을 모두 None으로 처리한다. JSON 출력 시 ensure_ascii=False로 한글을 그대로 보존하고 indent=2로 가독성을 높였으며, CSV 명세에 따라 인코딩은 utf-8을 명시했다. FileNotFoundError와 UnicodeDecodeError를 각각 별도로 잡아 파일 부재와 인코딩 오류를 구분해 기록한다.

In [2]:
class InvalidJamoError(ValueError):
    pass

def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"str 타입이 아닙니다: {type(c)}")
    if len(c) != 1:
        raise ValueError(f"길이가 1이 아닙니다: {repr(c)}")
    code = ord(c)
    if 0x3131 <= code <= 0x314E:
        return "자음"
    if 0x314F <= code <= 0x3163:
        return "모음"
    raise InvalidJamoError(f"한글 자모가 아닙니다: {repr(c)}")

inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        result = classify_jamo(item)
        print(f"{item!r} → {result}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")
    except TypeError as e:
        print(f"[TypeError] {e}")

'ㄱ' → 자음
'ㅏ' → 모음
'ㄲ' → 자음
[InvalidJamoError] 한글 자모가 아닙니다: '가'
[ValueError] 길이가 1이 아닙니다: 'AB'
[TypeError] str 타입이 아닙니다: <class 'int'>
'ㅎ' → 자음
'ㅣ' → 모음
[ValueError] 길이가 1이 아닙니다: ''


InvalidJamoError를 ValueError의 자식으로 정의한 이유는, 한글 자모가 아닌 문자 입력은 "값의 의미가 잘못된" 상황이므로 ValueError 계층에 속하는 것이 의미론적으로 적절하기 때문이다(Exception으로 만들면 기존 ValueError 핸들러가 잡지 못해 호환성이 떨어진다). except 절은 반드시 InvalidJamoError → ValueError 순으로 배치해야 하며, 순서가 바뀌면 부모 클래스가 자식 예외를 먼저 잡아버려 InvalidJamoError 분기에 도달하지 못한다.